# Cross-Device Performance · measure the edge, compare to the DGX

A benchmark number means nothing on its own. "356 GFLOP/s" is only useful next to another machine's number, on the same test, measured the same way. This lab writes one small benchmark, runs it on this DGX and on any other device you can reach, then merges the results and compares them: compute, memory bandwidth, interpreter speed, memory capacity, and power.

It is the payoff for the two data pre-labs. You run the benchmark with **uv** (from Lab CC, one portable script, its own clean environment) and you plot the comparison in the house style (from Lab DD, colorblind and grayscale safe). Where labs 02 and 06 profile the DGX alone, this one is the cross-device picture: a small NVIDIA edge board like the Jetson Thor next to a 130 GB unified-memory DGX Spark.

Work top to bottom. Run every code cell and read what comes back.

## How this notebook works

- **[Notebook cell]** runs here with **Shift+Enter**.
- **[Terminal step]** is a command you run on another machine (a Jetson, a laptop), then copy one small CSV back here.
- Results land in `~/perfLab/`: one `bench_<device>.csv` per machine, then a merged table and figures.
- The comparison always has a real reference: a DGX Spark baseline measured on the class box ships with the lab.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, checkpoints, and the figure helpers.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

### Preflight · check your environment

In [ ]:
preflight([
    check("numpy importable", pythonImportable("numpy"),
          hint="numpy is preinstalled in the class image."),
    check("pandas importable", pythonImportable("pandas"),
          hint="pandas is preinstalled in the class image."),
    check("matplotlib importable", pythonImportable("matplotlib"),
          hint="matplotlib is preinstalled in the class image."),
    check("uv on PATH", commandOnPath("uv"),
          hint="uv is baked into the class image; it runs the portable benchmark."),
    check("your home folder is writable", dirExists("~"),
          hint="You need a home folder to save results."),
], infoRows=[("nvidia-smi", "used for the GPU and power rows where available")])

---
## Part 1 · The portable benchmark

The whole comparison rests on running **the same test** everywhere. So the benchmark is one small file with no project to set up: an inline dependency block at the top (the PEP 723 format from Lab CC) tells `uv` it needs only numpy, and `uv run benchmark.py` builds that environment on the spot. The same command works on a Jetson, a laptop, or this DGX.

It measures three things every machine has, so nothing is skipped on a device without a GPU:

- **CPU compute** with a numpy matmul at a couple of sizes, reported as GFLOP/s (a matmul does `2*N**3` floating point operations).
- **Memory bandwidth** by copying a large array, reported as GB/s.
- **Interpreter overhead** with a tight Python loop, reported as million operations per second. This one barely changes with hardware and shows Python's fixed cost.

It also records the device identity (hostname, architecture, cores, RAM, GPU, power) so merged results stay labelled, and it adds a **GPU compute** row when a torch CUDA runtime is importable.

**[Notebook cell]** Write the benchmark into your lab folder. `%%writefile` saves the cell to a file (the path is literal, so we `cd` into `~/perfLab` first and write a plain name):

In [ ]:
from pathlib import Path
perf = Path.home() / "perfLab"
perf.mkdir(exist_ok=True)
%cd ~/perfLab

In [ ]:
%%writefile benchmark.py
# /// script
# requires-python = ">=3.9"
# dependencies = ["numpy>=1.24"]
# ///
"""Portable edge micro-benchmark. Runs ANYWHERE with `uv run benchmark.py`
(uv reads the inline dependency block above and builds a clean environment with
just numpy). It measures three things every device has, tags the result with
this machine's identity, adds a GPU row when a torch CUDA runtime happens to be
importable, and writes a tidy CSV plus JSON you can merge with other devices.

  1. CPU compute  - numpy matmul at a few sizes -> GFLOP/s (FLOPs = 2*N^3)
  2. Memory bandwidth - large array copy/sum -> GB/s
  3. Interpreter overhead - a tight Python loop -> million ops/s

Usage:  uv run benchmark.py [--device NAME] [--trials N] [--out DIR]
"""
import argparse
import csv
import json
import os
import platform
import re
import socket
import statistics
import subprocess
import sys
import time

import numpy as np


def sample_times(fn, trials, warmup=2):
    """Return a list of per-call seconds, discarding warmup runs."""
    for _ in range(warmup):
        fn()
    out = []
    for _ in range(trials):
        start = time.perf_counter()
        fn()
        out.append(time.perf_counter() - start)
    return out


def pctl(values, q):
    ordered = sorted(values)
    idx = min(len(ordered) - 1, int(round((q / 100.0) * (len(ordered) - 1))))
    return ordered[idx]


def cpu_model():
    try:
        with open("/proc/cpuinfo") as fh:
            text = fh.read()
        for key in ("model name", "Model", "Hardware", "cpu model"):
            m = re.search(rf"{key}\s*:\s*(.+)", text)
            if m:
                return m.group(1).strip()
    except Exception:
        pass
    return platform.processor() or platform.machine() or "unknown"


def total_ram_gb():
    try:
        pages = os.sysconf("SC_PHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return round(pages * page_size / 1e9, 1)
    except Exception:
        return None


def read_power_watts():
    """Best-effort GPU/board power in watts. Tries the standard NVIDIA driver
    first (DGX Spark and desktop GPUs report power.draw), then Tegra's
    tegrastats (Jetson). Returns None where no source is readable."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=power.draw", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5)
        if out.returncode == 0:
            val = out.stdout.strip().splitlines()[0].strip()
            if val and val.upper() not in ("N/A", "[N/A]"):
                return round(float(val), 2)
    except Exception:
        pass
    try:
        out = subprocess.run(["tegrastats", "--interval", "500"],
                             capture_output=True, text=True, timeout=3)
        text = out.stdout
    except Exception:
        text = ""
    m = re.search(r"(?:VDD_IN|POM_5V_IN|VDD_SYS_GPU)\s+(\d+)mW", text)
    if m:
        return round(int(m.group(1)) / 1000.0, 2)
    return None


def gpu_identity_and_matmul(n, trials):
    """(gpu_name, gflops_median) using torch if a CUDA/MPS runtime is importable.
    Under `uv run` the environment holds only numpy, so torch is absent and this
    returns (None, None) cleanly. Where torch exists (the class image), it times
    an on-device matmul."""
    try:
        import torch
    except Exception:
        return None, None
    if torch.cuda.is_available():
        dev, name = "cuda", torch.cuda.get_device_name(0)
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        dev, name = "mps", "Apple MPS"
    else:
        return None, None
    a = torch.rand(n, n, device=dev)
    b = torch.rand(n, n, device=dev)

    def once():
        c = a @ b
        if dev == "cuda":
            torch.cuda.synchronize()
        else:
            _ = c.cpu()
    times = sample_times(once, trials, warmup=3)
    flops = 2.0 * n ** 3
    return name, round(flops / statistics.median(times) / 1e9, 1)


def bench_cpu_matmul(sizes, trials):
    rows = []
    for n in sizes:
        a = np.random.rand(n, n)
        b = np.random.rand(n, n)
        times = sample_times(lambda: a @ b, trials)
        flops = 2.0 * n ** 3
        rows.append(dict(benchmark="cpu_matmul", size=n, unit="GFLOP/s",
                         value=round(flops / statistics.median(times) / 1e9, 1),
                         p95_value=round(flops / pctl(times, 95) / 1e9, 1)))
    return rows


def bench_mem_bandwidth(mb, trials):
    n = int(mb * 1e6 / 8)          # float64 elements
    a = np.random.rand(n)
    dst = np.empty_like(a)         # preallocated: measure steady-state copy, not malloc
    # a read+write copy touches 2*bytes; report GB/s of traffic moved.
    bytes_moved = 2 * a.nbytes
    times = sample_times(lambda: np.copyto(dst, a), trials)
    return [dict(benchmark="mem_bandwidth", size=mb, unit="GB/s",
                 value=round(bytes_moved / statistics.median(times) / 1e9, 1),
                 p95_value=round(bytes_moved / pctl(times, 95) / 1e9, 1))]


def bench_interpreter(iters, trials):
    def loop():
        total = 0
        for i in range(iters):
            total += i * i
        return total
    times = sample_times(loop, trials)
    return [dict(benchmark="interpreter", size=iters, unit="Mops/s",
                 value=round(iters / statistics.median(times) / 1e6, 1),
                 p95_value=round(iters / pctl(times, 95) / 1e6, 1))]


def main():
    ap = argparse.ArgumentParser(description="Portable edge micro-benchmark.")
    ap.add_argument("--device", default=os.environ.get("DEVICE_NAME"),
                    help="label for this machine (default: hostname)")
    ap.add_argument("--trials", type=int, default=12)
    ap.add_argument("--out", default=".", help="directory for the CSV/JSON")
    args = ap.parse_args()

    host = socket.gethostname().split(".")[0]
    gpu_name, gpu_gflops = gpu_identity_and_matmul(2048, max(4, args.trials // 2))
    device = args.device or host or (gpu_name and re.sub(r"[^A-Za-z0-9]+", "", gpu_name)) or "edge"

    meta = dict(
        device=device, hostname=host, arch=platform.machine(),
        cpu=cpu_model(), cores_logical=os.cpu_count(), ram_gb=total_ram_gb(),
        python=platform.python_version(), numpy=np.__version__,
        gpu=gpu_name or "none", power_w=read_power_watts(),
        omp_threads=os.environ.get("OMP_NUM_THREADS", "default"),
    )

    rows = []
    rows += bench_cpu_matmul([1024, 2048], args.trials)
    rows += bench_mem_bandwidth(256, args.trials)
    rows += bench_interpreter(2_000_000, max(5, args.trials // 2))
    if gpu_gflops is not None:
        rows.append(dict(benchmark="gpu_matmul", size=2048, unit="GFLOP/s",
                         value=gpu_gflops, p95_value=gpu_gflops))

    # tidy: every row carries the device identity so merged frames stay joinable
    tidy = [{**meta, **r} for r in rows]

    os.makedirs(args.out, exist_ok=True)
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", device)
    csv_path = os.path.join(args.out, f"bench_{stem}.csv")
    json_path = os.path.join(args.out, f"bench_{stem}.json")
    fields = list(tidy[0].keys())
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        w.writeheader()
        w.writerows(tidy)
    with open(json_path, "w") as fh:
        json.dump(dict(meta=meta, results=rows), fh, indent=2)

    print(f"device : {device}  ({meta['arch']}, {meta['cores_logical']} cores, "
          f"{meta['ram_gb']} GB RAM, gpu={meta['gpu']}, power={meta['power_w']} W)")
    for r in rows:
        print(f"  {r['benchmark']:<14} size={r['size']:<9} "
              f"{r['value']:>9} {r['unit']}  (p95 {r['p95_value']})")
    print(f"wrote {csv_path} and {json_path}")


if __name__ == "__main__":
    main()


**[Notebook cell]** Have a quick look at what you just wrote. This is the exact file you will carry to other machines:

In [ ]:
showFile("~/perfLab/benchmark.py", language="python", maxLines=40)

---
## Part 2 · Run it on this machine (the DGX)

Run the benchmark right here first. `uv` reads the inline dependencies, builds a clean numpy-only environment the first time (a few seconds), and runs the test. The `--device` label names this row in the merged table:

In [ ]:
import os
me = os.uname().nodename.split(".")[0]
print("this device will be labelled:", me)
!uv run benchmark.py --device {me} --trials 16

That printed a summary and wrote `bench_<device>.csv` and a matching `.json`. Notice the GPU row is absent here: `uv run` gave the script its own environment with **only numpy**, so torch is not visible even though the class image has it. That is uv doing its job, isolating the run. In Part 5 you will see how to include the GPU where you want it.

In [ ]:
import pandas as pd
mine = pd.read_csv(f"~/perfLab/bench_{me}.csv".replace("~", str(Path.home())))
mine[["device", "arch", "cores_logical", "ram_gb", "benchmark", "size", "value", "unit"]]

In [ ]:
checkpoint("Part 2 - benchmarked this machine", [
    check("benchmark.py written", fileExists("~/perfLab/benchmark.py"),
          hint="Run the %%writefile cell in Part 1."),
    check("a result CSV exists for this device",
          fileNonEmpty(f"~/perfLab/bench_{me}.csv", minLines=3),
          hint="Run the `uv run benchmark.py` cell above."),
], successNote="You have one real device measured. Next, get a second one so there is something to compare.")

---
## Part 3 · Get a second device

A comparison needs at least two machines. The point of the portable script is that the second one is easy.

**[Terminal step]** On another device you can reach (a Jetson Thor from the course fleet, or your own laptop):

1. Copy `benchmark.py` to it (a USB stick, `scp`, or paste the file).
2. Make sure `uv` is installed there: `curl -LsSf https://astral.sh/uv/install.sh | sh`
3. Run the same command, with a name for that device:

       uv run benchmark.py --device jetson-thor

4. Copy the `bench_jetson-thor.csv` it writes back into this machine's `~/perfLab/` folder (upload it in the Jupyter file browser).

The Jetson is the interesting comparison: same NVIDIA ARM and CUDA family as the DGX, but a much smaller machine. Two points on one line, at very different scale.

**[Notebook cell]** No second device handy this minute? Run this to drop in a small set of **example** rows so you can still build the comparison. These are clearly labelled illustrative numbers, not measurements of your hardware. Replace them with real runs whenever you can. The DGX Spark **baseline** written here is real, measured on the class box:

In [ ]:
import csv
FIELDS = ["device","hostname","arch","cpu","cores_logical","ram_gb","python","numpy",
          "gpu","power_w","omp_threads","benchmark","size","unit","value","p95_value"]

# Real DGX Spark baseline (measured on the class GB10 box with this benchmark).
dgx_baseline = [
    dict(device="dgx-spark-baseline", hostname="cs494", arch="aarch64", cpu="aarch64", cores_logical=20,
         ram_gb=130.6, python="3.12.3", numpy="2.5.1", gpu="none", power_w=4.18, omp_threads="default",
         benchmark="cpu_matmul", size=1024, unit="GFLOP/s", value=397.8, p95_value=373.4),
    dict(device="dgx-spark-baseline", hostname="cs494", arch="aarch64", cpu="aarch64", cores_logical=20,
         ram_gb=130.6, python="3.12.3", numpy="2.5.1", gpu="none", power_w=4.18, omp_threads="default",
         benchmark="cpu_matmul", size=2048, unit="GFLOP/s", value=355.7, p95_value=353.8),
    dict(device="dgx-spark-baseline", hostname="cs494", arch="aarch64", cpu="aarch64", cores_logical=20,
         ram_gb=130.6, python="3.12.3", numpy="2.5.1", gpu="none", power_w=4.18, omp_threads="default",
         benchmark="mem_bandwidth", size=256, unit="GB/s", value=48.7, p95_value=48.2),
    dict(device="dgx-spark-baseline", hostname="cs494", arch="aarch64", cpu="aarch64", cores_logical=20,
         ram_gb=130.6, python="3.12.3", numpy="2.5.1", gpu="none", power_w=4.18, omp_threads="default",
         benchmark="interpreter", size=2000000, unit="Mops/s", value=33.6, p95_value=33.6),
]

# Illustrative example peers (NOT measured on your hardware - replace with real runs).
example_peers = [
    dict(device="jetson-thor-example", hostname="example", arch="aarch64", cpu="ARM Cortex",
         cores_logical=12, ram_gb=64.0, python="3.10", numpy="1.26", gpu="NVIDIA Thor (example)",
         power_w=40.0, omp_threads="default", benchmark=b, size=s, unit=u, value=v, p95_value=v)
    for (b, s, u, v) in [("cpu_matmul",1024,"GFLOP/s",95.0),("cpu_matmul",2048,"GFLOP/s",88.0),
                         ("mem_bandwidth",256,"GB/s",34.0),("interpreter",2000000,"Mops/s",14.0),
                         ("gpu_matmul",2048,"GFLOP/s",5200.0)]
] + [
    dict(device="laptop-x86-example", hostname="example", arch="x86_64", cpu="Intel Core",
         cores_logical=8, ram_gb=16.0, python="3.11", numpy="1.26", gpu="none",
         power_w="", omp_threads="default", benchmark=b, size=s, unit=u, value=v, p95_value=v)
    for (b, s, u, v) in [("cpu_matmul",1024,"GFLOP/s",180.0),("cpu_matmul",2048,"GFLOP/s",150.0),
                         ("mem_bandwidth",256,"GB/s",22.0),("interpreter",2000000,"Mops/s",48.0)]
]

for name, rows in [("dgx-spark-baseline", dgx_baseline),
                   ("jetson-thor-example", example_peers[:5]),
                   ("laptop-x86-example", example_peers[5:])]:
    path = perf / f"bench_{name}.csv"
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS)
        w.writeheader(); w.writerows(rows)
    print("wrote", path.name)

---
## Part 4 · Merge the devices

Every machine wrote the same columns, so merging is just stacking the CSVs. This is the tidy-data payoff from Lab CC: one row per (device, benchmark), every row carrying its device identity, so a comparison is a `groupby` away.

In [ ]:
import pandas as pd, glob
frames = [pd.read_csv(p) for p in sorted(glob.glob(str(perf / "bench_*.csv")))]
alldev = pd.concat(frames, ignore_index=True)
devices = sorted(alldev["device"].unique())
print(len(frames), "device files,", len(devices), "devices:", devices)
alldev.pivot_table(index="device", columns="benchmark", values="value").round(1)

In [ ]:
checkpoint("Part 4 - devices merged", [
    check("at least two devices to compare",
          lambda: (alldev["device"].nunique() >= 2,
                   f"{alldev['device'].nunique()} devices: {devices}"),
          hint="Run the example-data cell in Part 3, or add a real second device's CSV."),
    check("cpu_matmul present for every device",
          lambda: (alldev[alldev.benchmark=='cpu_matmul']['device'].nunique() == alldev['device'].nunique(),
                   "cpu_matmul is the common axis"),
          hint="Every device runs cpu_matmul; re-run any device missing it."),
], successNote="One tidy frame, several devices, ready to plot.")

---
## Part 5 · Compare

Now the figures. Apply the house style once, then make one chart per question. Each saves to `~/perfLab/figures/` as PDF and PNG, the same as Lab DD.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
palette = applyHouseStyle()
order = sorted(devices, key=lambda d: alldev.loc[(alldev.device==d)&(alldev.benchmark=="cpu_matmul"),"value"].max())
colors = {d: palette[i % len(palette)] for i, d in enumerate(order)}
print("devices, slowest to fastest CPU:", order)

**[Notebook cell]** **Which machine computes fastest?** A grouped bar of CPU GFLOP/s per device, one group per problem size. Bigger is faster:

In [ ]:
cpu = alldev[alldev.benchmark == "cpu_matmul"]
sizes = sorted(cpu["size"].unique())
x = np.arange(len(sizes)); w = 0.8 / len(order)
fig, ax = plt.subplots()
for i, d in enumerate(order):
    vals = [cpu[(cpu.device==d)&(cpu["size"]==s)]["value"].mean() for s in sizes]
    ax.bar(x + i*w, vals, w, label=d, color=colors[d])
ax.set_xticks(x + w*(len(order)-1)/2); ax.set_xticklabels([f"{s}x{s}" for s in sizes])
ax.set_xlabel("matrix size"); ax.set_ylabel("CPU compute (GFLOP/s)")
ax.set_title("CPU compute by device"); ax.legend(title="device")
saveFigure(fig, "cpu_compute_by_device"); plt.show()

**[Notebook cell]** **How much faster is the DGX?** Pick the DGX row as the reference and compute a speedup for each device on each common benchmark. A speedup table is often the clearest single result in a report:

In [ ]:
ref = next((d for d in order if "dgx" in d.lower()), order[-1])
wide = alldev.pivot_table(index="device", columns="benchmark", values="value")
common = [c for c in ["cpu_matmul","mem_bandwidth","interpreter"] if c in wide.columns]
speedup = (wide[common] / wide.loc[ref, common]).round(2)
print("reference device:", ref)
speedup.rename(columns=lambda c: c + " (x vs ref)")

**[Notebook cell]** **What can it hold?** Speed is not the only axis. The DGX Spark's 130 GB of unified memory (one pool shared by CPU and GPU) runs models a small board or a laptop cannot fit at all. Compare total RAM per device. This capacity gap is a different kind of advantage from raw speed:

In [ ]:
ram = alldev.groupby("device")["ram_gb"].max().reindex(order)
fig, ax = plt.subplots()
ax.barh(ram.index, ram.values, color=[colors[d] for d in ram.index])
for i, v in enumerate(ram.values):
    ax.text(v, i, f" {v:g} GB", va="center", fontsize=9)
ax.set_xlabel("total memory (GB)"); ax.set_title("Memory capacity by device")
saveFigure(fig, "memory_capacity_by_device"); plt.show()

**[Notebook cell]** **The GPU dimension (optional).** `uv run` left torch out, so the CPU comparison above is GPU-blind. To add this DGX's GPU, run the same script with the class image's own Python, which does have torch and CUDA. It writes a `dgx-gpu` row you can merge. If torch is missing, the script simply skips the GPU row, so this is safe to run anywhere:

In [ ]:
!python benchmark.py --device dgx-gpu --trials 12 || echo "(no torch/CUDA here - GPU row skipped, which is fine)"
# reload and show any gpu_matmul rows now present across all devices
alldev = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(str(perf / "bench_*.csv")))], ignore_index=True)
gpu = alldev[alldev.benchmark == "gpu_matmul"]
gpu[["device","size","value","unit","gpu"]] if len(gpu) else "No GPU rows yet - run on a device with torch/CUDA."

In [ ]:
checkpoint("Part 5 - comparison figures", [
    check("CPU compute chart saved", fileExists("~/perfLab/figures/cpu_compute_by_device.pdf"),
          hint="Run the grouped-bar cell."),
    check("memory-capacity chart saved", fileExists("~/perfLab/figures/memory_capacity_by_device.pdf"),
          hint="Run the memory-capacity cell."),
], successNote="Compute, speedup, and capacity: three different ways one machine can beat another.")

---
## Part 6 · Sustained load and thermal behaviour

A single fast run can hide the real story. Small edge devices heat up under a steady load and **throttle**, dropping their clock to stay cool, so their sustained speed is lower than their burst speed. A well-cooled machine like the DGX holds its rate. This is the same idea as Lab 06 Part 7, now measured directly.

**[Notebook cell]** Run a matmul in a loop for about 30 seconds on this machine, timing each pass to get throughput over time, and sampling GPU temperature and power from `nvidia-smi` alongside it. Watch whether the throughput line stays flat (no throttling) or sags:

In [ ]:
import time, subprocess, numpy as np
def sample_gpu():
    try:
        out = subprocess.run(["nvidia-smi",
            "--query-gpu=temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=3)
        temp, power = (out.stdout.strip().split("\n")[0].split(","))
        return float(temp), float(power)
    except Exception:
        return None, None

N, seconds = 2048, 30
a = np.random.rand(N, N); b = np.random.rand(N, N)
flops = 2.0 * N**3
t_end = time.time() + seconds
series = []
while time.time() < t_end:
    t0 = time.perf_counter(); a @ b; dt = time.perf_counter() - t0
    temp, power = sample_gpu()
    series.append(dict(t=round(time.time() - (t_end - seconds), 1),
                       gflops=round(flops/dt/1e9, 1), temp_c=temp, power_w=power))
sus = pd.DataFrame(series)
print(f"{len(sus)} passes in ~{seconds}s; "
      f"throughput first vs last 5: {sus['gflops'][:5].mean():.0f} -> {sus['gflops'][-5:].mean():.0f} GFLOP/s")
sus.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(sus["t"], sus["gflops"], color=palette[0], marker=FIGURE_MARKERS[0], markevery=5)
ax.set_xlabel("time (s)"); ax.set_ylabel("throughput (GFLOP/s)")
ax.set_title("Sustained CPU throughput on this machine")
if sus["temp_c"].notna().any():
    ax2 = ax.twinx()
    ax2.plot(sus["t"], sus["temp_c"], color=palette[1], linestyle="--")
    ax2.set_ylabel("GPU temp (C)", color=palette[1]); ax2.grid(False)
saveFigure(fig, "sustained_throughput"); plt.show()

In [ ]:
held = sus["gflops"][-5:].mean() / sus["gflops"][:5].mean()
checkpoint("Part 6 - sustained load", [
    check("sustained series collected", lambda: (len(sus) > 5, f"{len(sus)} timed passes"),
          hint="Run the 30-second loop cell."),
    check("sustained-throughput figure saved", fileExists("~/perfLab/figures/sustained_throughput.pdf"),
          hint="Run the plot cell."),
], successNote=f"This machine held {held*100:.0f}% of its starting throughput. "
               "On a small board under the same load you would often see that drop further.")

### Lab scorecard

In [ ]:
labSummary("Cross-Device Performance")

---

You wrote one portable benchmark, ran it with uv on more than one machine, merged the results with tidy data, and turned them into figures that compare compute, memory capacity, and thermal behaviour. That is the full arc of the on-ramp put to work: drive the machine (**AA**), Python (**BB**), a clean reproducible experiment (**CC**), publication figures (**DD**), and now a real cross-device measurement you could defend in a report.

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Cross-Device Performance")